# Getting started with our ultimate beginner guide!

## This tutorial will walk you through the basics of using the `usb` lighting package. Let's get started by training a FixMatch model on Medmnist!

In [12]:
import semilearn
from semilearn import get_dataset, get_data_loader, get_net_builder, get_algorithm, get_config, Trainer
import subprocess
import os

result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value

## Step 1: define configs and create config

In [19]:
config = {
    'algorithm': 'fixmatch',
    'net': 'vit_tiny_patch2_32',
    'use_pretrain': True,
    'pretrain_path': 'https://github.com/microsoft/Semi-supervised-learning/releases/download/v.0.0.0/vit_tiny_patch2_32_mlp_im_1k_32.pth',

    'epoch': 1,
    'num_train_iter': 5000,
    'num_eval_iter': 500,
    'num_log_iter': 50,
    'optim': 'AdamW',
    'lr': 5e-4,
    'layer_decay': 0.5,
    'batch_size': 16,
    'eval_batch_size': 16,

    'dataset': 'tissuemnist',
    'num_classes': 8,   # ⚠️ 这里要和 tissuemnist 的真实类别数一致
    'num_labels': 40,
    'img_size': 32,
    'crop_ratio': 0.875,
    'data_dir': '/root/autodl-tmp/Semi-supervised-learning/notebooks/data/medmnist',#从./data改成绝对路径了

    # ✅ 关键修复：给无标注数量（可用全部剩余）
    # 'ulb_num_labels': 49960,

    'hard_label': True,
    'uratio': 2,
    'ulb_loss_ratio': 1.0,

    'gpu': 0,
    'world_size': 1,
    'distributed': False,
    "num_workers": 2,
    
    'include_lb_to_ulb': False,
    
    # IMPORTANT
    'amp': False,
    'seed': 0,
}
config = get_config(config)
print(config)

Namespace(save_dir='./saved_models', save_name='fixmatch', resume=False, load_path=None, overwrite=True, use_tensorboard=False, use_wandb=False, use_aim=False, epoch=1, num_train_iter=5000, num_warmup_iter=0, num_eval_iter=500, num_log_iter=50, num_labels=40, batch_size=16, uratio=2, eval_batch_size=16, ema_m=0.999, ulb_loss_ratio=1.0, optim='AdamW', lr=0.0005, momentum=0.9, weight_decay=0.0005, layer_decay=0.5, net='vit_tiny_patch2_32', net_from_name=False, use_pretrain=True, pretrain_path='https://github.com/microsoft/Semi-supervised-learning/releases/download/v.0.0.0/vit_tiny_patch2_32_mlp_im_1k_32.pth', algorithm='fixmatch', use_cat=True, use_amp=False, clip_grad=0, imb_algorithm=None, data_dir='/root/autodl-tmp/Semi-supervised-learning/notebooks/data/medmnist', dataset='tissuemnist', num_classes=8, train_sampler='RandomSampler', num_workers=2, include_lb_to_ulb=False, lb_imb_ratio=1, ulb_imb_ratio=1, ulb_num_labels=None, img_size=32, crop_ratio=0.875, max_length=512, max_length_se

## Step 2: create model and specify algorithm

In [20]:
algorithm = get_algorithm(config, get_net_builder(config.net, from_name=False), tb_log=None, logger=None)

RuntimeError: Failed to setup the default `root` directory. Please specify and create the `root` directory manually.

## Step 3: create dataset

In [9]:
dataset_dict = get_dataset(cfg, cfg.algorithm, cfg.dataset, cfg.num_labels, cfg.num_classes,
                           data_dir=cfg.data_dir, include_lb_to_ulb=cfg.include_lb_to_ulb)

print(dataset_dict.keys())  # 应该有 train_lb train_ulb eval test

train_lb_loader  = get_data_loader(cfg, dataset_dict['train_lb'], cfg.batch_size)
train_ulb_loader = get_data_loader(cfg, dataset_dict['train_ulb'], int(cfg.batch_size * cfg.uratio))
eval_loader      = get_data_loader(cfg, dataset_dict['eval'], cfg.eval_batch_size)


_IncompatibleKeys(missing_keys=['head.weight', 'head.bias'], unexpected_keys=[])
Create optimizer and scheduler


AttributeError: 'NoneType' object has no attribute 'keys'

## Step 4: train

In [1]:
trainer = Trainer(config, algorithm)
trainer.fit(train_lb_loader, train_ulb_loader, eval_loader)

NameError: name 'Trainer' is not defined

## Step 5: evaluate

In [6]:
trainer.evaluate(eval_loader)

[2026-01-22 17:21:00,018 INFO] confusion matrix
[2026-01-22 17:21:00,020 INFO] [[0.955 0.002 0.002 0.001 0.    0.    0.001 0.    0.029 0.01 ]
 [0.    0.993 0.    0.    0.    0.    0.    0.    0.    0.007]
 [0.05  0.    0.801 0.012 0.096 0.006 0.01  0.023 0.001 0.001]
 [0.011 0.001 0.006 0.895 0.012 0.047 0.013 0.011 0.003 0.001]
 [0.001 0.    0.001 0.007 0.949 0.    0.004 0.037 0.001 0.   ]
 [0.002 0.    0.001 0.036 0.008 0.915 0.002 0.036 0.    0.   ]
 [0.004 0.    0.005 0.001 0.002 0.002 0.985 0.    0.001 0.   ]
 [0.003 0.001 0.003 0.003 0.007 0.01  0.    0.973 0.    0.   ]
 [0.009 0.009 0.    0.    0.    0.    0.    0.    0.978 0.004]
 [0.004 0.033 0.    0.    0.    0.    0.    0.    0.003 0.96 ]]
[2026-01-22 17:21:00,022 INFO] evaluation metric
[2026-01-22 17:21:00,023 INFO] acc: 0.9404
[2026-01-22 17:21:00,024 INFO] precision: 0.9418
[2026-01-22 17:21:00,026 INFO] recall: 0.9404
[2026-01-22 17:21:00,026 INFO] f1: 0.9398


{'acc': 0.9404,
 'precision': 0.9417930516155932,
 'recall': 0.9404,
 'f1': 0.9398139982848557}

## Step 6: predict

In [7]:
y_pred, y_logits = trainer.predict(eval_loader)